# Практична робота №10 — Багатокритеріальна оптимізація: множина Парето та методи згортки

Вкажи номер варіанта, коли клітинка попросить — решта коду підхопить твої критерії f1, f2, обмеження на ресурси, мінімальний план випуску та ваги критеріїв автоматично.

In [ ]:
import numpy as np
import pandas as pd
import itertools
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import linprog

In [ ]:
from variants import VARIANTS

_default = globals().get("VARIANT", 1)
_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()
VARIANT = int(_raw) if _raw else _default
assert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}"

## 1. Побудова моделі

З таблиці варіантів дістаємо коефіцієнти прибутку `f1` (максимізуємо), коефіцієнти витрат/викидів `f2` (мінімізуємо), обмеження на два ресурси (`<=`), мінімальний план випуску `x1+x2 >= plan` і ваги критеріїв `w1, w2`. Усю задачу зводимо до стандартної форми лінійного програмування `scipy.optimize.linprog` (яка завжди мінімізує, тому максимізацію f1 реалізуємо як мінімізацію -f1).

In [ ]:
f1_c, f2_c, res, plan, weights = VARIANTS[VARIANT]
w1, w2 = weights

A_ub = [[p, q] for (p, q, _) in res] + [[-1, -1]]   # останній рядок: -(x1+x2) <= -plan
b_ub = [r for (_, _, r) in res] + [-plan]
bounds = [(0, None), (0, None)]

f1 = lambda x: f1_c[0] * x[0] + f1_c[1] * x[1]   # прибуток -> max
f2 = lambda x: f2_c[0] * x[0] + f2_c[1] * x[1]   # витрати/викиди -> min

print(f"Варіант {VARIANT}")
print(f"f1(x1,x2) = {f1_c[0]}*x1 + {f1_c[1]}*x2  -> max")
print(f"f2(x1,x2) = {f2_c[0]}*x1 + {f2_c[1]}*x2  -> min")
for p, q, r in res:
    print(f"обмеження ресурсу: {p}*x1 + {q}*x2 <= {r}")
print(f"мінімальний план: x1 + x2 >= {plan}")
print(f"ваги критеріїв: w1 = {w1}, w2 = {w2}")

## 2. Оптимуми за кожним критерієм окремо. Точка ідеалу і надиру

Розв'язуємо дві однокритеріальні задачі ЛП: максимізація f1 і мінімізація f2. Для задачі з рівно двома критеріями ця пара («платіжна таблиця») дає точне значення точки ідеалу (найкраще f1, найкраще f2 окремо) і точне значення точки надиру (найгірше значення кожного критерію **на множині Парето** — воно досягається саме в точці оптимуму іншого критерію).

In [ ]:
r1 = linprog([-f1_c[0], -f1_c[1]], A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')  # max f1
r2 = linprog([f2_c[0], f2_c[1]], A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')      # min f2
assert r1.success and r2.success, "задача ЛП несумісна для цього варіанта"

x_f1opt, x_f2opt = r1.x, r2.x
f1_ideal, f2_at_f1opt = f1(x_f1opt), f2(x_f1opt)
f1_at_f2opt, f2_ideal = f1(x_f2opt), f2(x_f2opt)
f1_nadir, f2_nadir = f1_at_f2opt, f2_at_f1opt

print(f"оптимум за f1: x = ({x_f1opt[0]:.3f}, {x_f1opt[1]:.3f}), f1 = {f1_ideal:.2f}, f2 = {f2_at_f1opt:.2f}")
print(f"оптимум за f2: x = ({x_f2opt[0]:.3f}, {x_f2opt[1]:.3f}), f1 = {f1_at_f2opt:.2f}, f2 = {f2_ideal:.2f}")
print(f"точка ідеалу  (f1_max, f2_min) = ({f1_ideal:.2f}, {f2_ideal:.2f}) — недосяжна")
print(f"точка надиру  (f1_min, f2_max) = ({f1_nadir:.2f}, {f2_nadir:.2f})")

## 3. Допустима область у просторі змінних і множина Парето

Допустима область — многокутник, обмежений двома ресурсними прямими, прямою плану і осями координат. Знаходимо всі його вершини перетином кожної пари граничних прямих і відбором тих перетинів, що справді задовольняють усі обмеження. Множина Парето в просторі змінних для двокритеріальної задачі ЛП — це єдине ребро многокутника між вершиною-оптимумом f1 і вершиною-оптимумом f2 (перевіряємо це нижче через метод ε-обмежень, а не просто порівнянням вершин — порівняння лише вершин може помилково визнати ефективною вершину, яку насправді домінує внутрішня точка ребра).

In [ ]:
lines = [(p, q, r) for (p, q, r) in res] + [(1, 1, plan), (1, 0, 0), (0, 1, 0)]
cons = [(p, q, r, 'le') for (p, q, r) in res] + [(1, 1, plan, 'ge'), (1, 0, 0, 'ge'), (0, 1, 0, 'ge')]

def feasible(pt, tol=1e-6):
    x1, x2 = pt
    for p, q, r, sense in cons:
        val = p * x1 + q * x2
        if sense == 'le' and val > r + tol:
            return False
        if sense == 'ge' and val < r - tol:
            return False
    return True

verts = set()
for (p1, q1, r1_), (p2, q2, r2_) in itertools.combinations(lines, 2):
    A = np.array([[p1, q1], [p2, q2]])
    if abs(np.linalg.det(A)) < 1e-9:
        continue
    pt = np.linalg.solve(A, np.array([r1_, r2_]))
    if feasible(pt):
        verts.add((round(float(pt[0]), 6), round(float(pt[1]), 6)))

center = np.mean(list(verts), axis=0)
verts_sorted = sorted(verts, key=lambda p: np.arctan2(p[1] - center[1], p[0] - center[0]))
poly = np.array(verts_sorted + [verts_sorted[0]])

fig0, ax0 = plt.subplots(figsize=(6, 5))
ax0.fill(poly[:, 0], poly[:, 1], color='lightblue', alpha=.6, label='допустима область X')
ax0.plot(poly[:, 0], poly[:, 1], 'b-', lw=1)
ax0.plot([x_f2opt[0], x_f1opt[0]], [x_f2opt[1], x_f1opt[1]], 'r-', lw=3, label='множина Парето')
ax0.plot(*x_f1opt, 'g^', ms=10, label='opt f1 (max прибуток)')
ax0.plot(*x_f2opt, 'mv', ms=10, label='opt f2 (min витрати)')
ax0.set_xlabel('x1'); ax0.set_ylabel('x2'); ax0.legend(fontsize=8)
ax0.set_title(f'Варіант {VARIANT}: допустима область та множина Парето')
fig0.tight_layout()
fig0.savefig('feasible_region.png', dpi=150)
fig0

## 4. Метод зважених сум (із нормалізацією критеріїв)

Переходимо до двох мінімізаційних критеріїв: g1 = -f1 (менше — краще), g2 = f2. Нормалізуємо кожен лінійно за діапазоном [ідеал; надир], отриманим на кроці 2, і для кожної ваги w із сітки 0; 0,1; ...; 1 розв'язуємо задачу ЛП `min w*g1_norm + (1-w)*g2_norm`. Нормалізація — афінне перетворення, тому задача залишається лінійною.

In [ ]:
g1_lo, g1_hi = -f1_ideal, -f1_nadir
g2_lo, g2_hi = f2_ideal, f2_nadir

ws_rows = []
for w in np.round(np.linspace(0, 1, 11), 2):
    c1 = w / (g1_hi - g1_lo)
    c2 = (1 - w) / (g2_hi - g2_lo)
    c = [c1 * (-f1_c[0]) + c2 * f2_c[0], c1 * (-f1_c[1]) + c2 * f2_c[1]]
    r = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    x = r.x
    ws_rows.append(dict(w=float(w), x1=float(x[0]), x2=float(x[1]), f1=float(f1(x)), f2=float(f2(x))))

ws_df = pd.DataFrame(ws_rows)
ws_points = {(round(row['f1'], 3), round(row['f2'], 3)) for row in ws_rows}
print(f"різних Парето-точок знайдено методом зважених сум: {len(ws_points)} з 11 запусків")
ws_df

## 5. Метод ε-обмежень

Критерій f1 лишаємо головним (максимізуємо), а f2 переводимо в обмеження `f2(x) <= eps`. Пускаємо не менше шести значень `eps` рівномірно між ідеальним і надирним значенням f2 — кожен запуск дає ще одну точку фронту Парето, причому ε-обмеження здатне знайти будь-яку точку, а не лише вершини опуклої оболонки.

In [ ]:
eps_list = np.round(np.linspace(f2_ideal, f2_nadir, 6), 3)
eps_rows = []
for eps in eps_list:
    A2 = A_ub + [[f2_c[0], f2_c[1]]]
    b2 = b_ub + [eps]
    r = linprog([-f1_c[0], -f1_c[1]], A_ub=A2, b_ub=b2, bounds=bounds, method='highs')
    if r.success:
        x = r.x
        eps_rows.append(dict(eps=float(eps), x1=float(x[0]), x2=float(x[1]), f1=float(f1(x)), f2=float(f2(x))))

eps_df = pd.DataFrame(eps_rows)
eps_points = {(round(row['f1'], 3), round(row['f2'], 3)) for row in eps_rows}
print(f"різних Парето-точок знайдено методом ε-обмежень: {len(eps_points)} з {len(eps_list)} запусків")
eps_df

## 6. Порівняння покриття фронту

Будуємо щільну (контрольну) розгортку методом ε-обмежень — 200 значень eps — як наближення істинного фронту Парето, і перевіряємо, що жодна з 200 точок не домінується жодною іншою (справжній непарето-оптимальний розв'язок там не мав би з'явитися). На цьому ж графіку показуємо, які саме точки знайшли метод зважених сум і метод ε-обмежень.

In [ ]:
eps_fine = np.linspace(f2_ideal, f2_nadir, 200)
front_fine = []
for eps in eps_fine:
    A2 = A_ub + [[f2_c[0], f2_c[1]]]
    b2 = b_ub + [eps]
    r = linprog([-f1_c[0], -f1_c[1]], A_ub=A2, b_ub=b2, bounds=bounds, method='highs')
    if r.success:
        front_fine.append((float(f1(r.x)), float(f2(r.x))))
front_fine = np.array(front_fine)

# перевірка: серед точок щільного фронту жодна не домінується іншою
dominated = 0
for i in range(len(front_fine)):
    f1i, f2i = front_fine[i]
    beats = (front_fine[:, 0] >= f1i - 1e-6) & (front_fine[:, 1] <= f2i + 1e-6) & \
            ((front_fine[:, 0] > f1i + 1e-6) | (front_fine[:, 1] < f2i - 1e-6))
    if beats.any():
        dominated += 1
print(f"домінованих точок серед {len(front_fine)} точок щільного фронту: {dominated} (має бути 0)")

fig1, ax1 = plt.subplots(figsize=(7, 5.5))
ax1.plot(front_fine[:, 0], front_fine[:, 1], 'k-', lw=2, label='фронт Парето (щільна розгортка ε)')
ax1.plot(ws_df['f1'], ws_df['f2'], 'rx', ms=10, mew=2, label='метод зважених сум')
ax1.plot(eps_df['f1'], eps_df['f2'], 'bo', ms=7, mfc='none', label='метод ε-обмежень')
ax1.plot(f1_ideal, f2_ideal, 'g*', ms=16, label='точка ідеалу')
ax1.plot(f1_nadir, f2_nadir, 'm*', ms=16, label='точка надиру')
ax1.set_xlabel('f1 — прибуток (max)'); ax1.set_ylabel('f2 — витрати/викиди (min)')
ax1.set_title(f'Варіант {VARIANT}: фронт Парето у просторі критеріїв')
ax1.legend(fontsize=8)
fig1.tight_layout()
fig1.savefig('pareto_front.png', dpi=150)
fig1

## 7. Лексикографічний метод

Головний критерій f1 оптимізуємо першим; дозволяємо відхилення δ = 5% від знайденого оптимуму (`f1 >= (1-δ)*f1_ideal`) і в межах цього допуску мінімізуємо f2 — це дає компромісну точку, відмінну від "жадібного" оптимуму лише за f1.

In [ ]:
delta = 0.05
A_lex = A_ub + [[-f1_c[0], -f1_c[1]]]
b_lex = b_ub + [-(1 - delta) * f1_ideal]
r_lex = linprog([f2_c[0], f2_c[1]], A_ub=A_lex, b_ub=b_lex, bounds=bounds, method='highs')
x_lex = r_lex.x

print(f"δ = {delta:.0%} від f1_ideal = {f1_ideal:.2f}  ->  f1 >= {(1 - delta) * f1_ideal:.2f}")
print(f"лексикографічний розв'язок: x = ({x_lex[0]:.3f}, {x_lex[1]:.3f}), "
      f"f1 = {f1(x_lex):.2f}, f2 = {f2(x_lex):.2f}")
print(f"для порівняння — 'жадібний' оптимум за f1: f1 = {f1_ideal:.2f}, f2 = {f2_at_f1opt:.2f}")

## 8. Увігнутий фронт: межа методу зважених сум

Штучна задача (обидва критерії мінімізуємо, щоб не ускладнювати знак): f1 = x → min, f2 = 1 - x² → min, x ∈ [0; 1]. Увесь відрізок кривої — Парето-оптимальний (будь-яке покращення одного критерію погіршує інший), але лінія рівня зваженої суми — пряма, і її оптимум на увігнутій дузі завжди "перестрибує" у один із двох кінців відрізка.

In [ ]:
xs = np.linspace(0, 1, 201)
F = np.c_[xs, 1 - xs ** 2]   # (f1, f2) для кожного x; обидва -> min

found = set()
for w in np.linspace(0, 1, 101):
    k = int(np.argmin(w * F[:, 0] + (1 - w) * F[:, 1]))   # min(w*f1 + (1-w)*f2)
    found.add(round(float(xs[k]), 3))

print(f"усього точок на увігнутому фронті: {len(xs)}")
print(f"метод зважених сум (101 вага) знайшов лише {len(found)}: {sorted(found)}")

fig2, ax2 = plt.subplots(figsize=(6, 5))
ax2.plot(F[:, 0], F[:, 1], 'k-', lw=2, label='увігнутий фронт Парето (усі точки ефективні)')
found_idx = [int(np.argmin(np.abs(xs - fx))) for fx in sorted(found)]
ax2.plot(F[found_idx, 0], F[found_idx, 1], 'rx', ms=14, mew=3, label='знайдено методом зважених сум')
ax2.set_xlabel('f1 (min)'); ax2.set_ylabel('f2 (min)')
ax2.set_title('Штучна задача з увігнутим фронтом')
ax2.legend(fontsize=8)
fig2.tight_layout()
fig2.savefig('concave_front.png', dpi=150)
fig2

## 9. TOPSIS для дискретного вибору альтернатив

Беремо п'ять точок фронту, знайдених методом ε-обмежень (крок 5), як п'ять дискретних альтернатив, і впорядковуємо їх методом TOPSIS із вагами свого варіанта: нормалізація матриці рішень → зважування → ідеальна/антиідеальна альтернатива → відстані → коефіцієнт близькості Cᵢ.

In [ ]:
alts = eps_df[['f1', 'f2']].values[:5]
names = [f"A{i+1}" for i in range(len(alts))]
w_vec = np.array([w1, w2])
benefit = np.array([True, False])   # f1 - чим більше тим краще; f2 - навпаки

N = alts / np.sqrt((alts ** 2).sum(axis=0))      # 1. нормалізація
V = N * w_vec                                    # 2. зважування
ideal_v = np.where(benefit, V.max(axis=0), V.min(axis=0))   # 3. ідеальна/антиідеальна
anti_v = np.where(benefit, V.min(axis=0), V.max(axis=0))
d_pos = np.linalg.norm(V - ideal_v, axis=1)       # 4. відстані
d_neg = np.linalg.norm(V - anti_v, axis=1)
C = d_neg / (d_pos + d_neg)                       # 5. коефіцієнт близькості
ranks = (-C).argsort().argsort() + 1

topsis_df = pd.DataFrame({
    'альтернатива': names, 'f1': alts[:, 0], 'f2': alts[:, 1],
    'd+': d_pos, 'd-': d_neg, 'C': C, 'ранг': ranks,
})
print(f"найкраща альтернатива: {names[int(np.argmax(C))]}")
topsis_df

## 10. Чутливість рангів TOPSIS до ваг критеріїв

Змінюємо вагу w1 на ±0,1 (друга вага доповнює до 1) і перевіряємо, чи лишається лідер ранжування тим самим.

In [ ]:
sens_rows = []
for dw in (-0.1, 0.0, 0.1):
    w1s = float(np.clip(w1 + dw, 0, 1)); w2s = 1 - w1s
    Vs = N * np.array([w1s, w2s])
    ideal_s = np.where(benefit, Vs.max(axis=0), Vs.min(axis=0))
    anti_s = np.where(benefit, Vs.min(axis=0), Vs.max(axis=0))
    d_pos_s = np.linalg.norm(Vs - ideal_s, axis=1)
    d_neg_s = np.linalg.norm(Vs - anti_s, axis=1)
    Cs = d_neg_s / (d_pos_s + d_neg_s)
    leader = names[int(np.argmax(Cs))]
    sens_rows.append(dict(dw=dw, w1=w1s, w2=w2s, leader=leader,
                           **{f"C_{n}": float(c) for n, c in zip(names, Cs)}))

sens_df = pd.DataFrame(sens_rows)
sens_df